In [1]:
%load_ext autoreload
%autoreload all
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

%load_ext autoreload
%autoreload all
from agent_harness import Agent, tool, PromptRegistry
from agent_harness.prompts import PromptRegistry


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
@tool(
    description="Convert a temperature from Celsius to Fahrenheit.",
    parameters={"celsius": {"type": "number", "description": "Temperature in Celsius."}},
    required=["celsius"],
)
def celsius_to_fahrenheit(celsius: float) -> str:
    return str(celsius * 9 / 5 + 32)


In [4]:
from agent_harness import tools
tools.TOOL_REGISTRY

{'calculate': <function agent_harness.tools.calculate(expression: str) -> str>,
 'search_web': <function agent_harness.tools.search_web(query: str) -> str>,
 'celsius_to_fahrenheit': <function __main__.celsius_to_fahrenheit(celsius: float) -> str>}

In [11]:
default_registry = PromptRegistry()

default_registry.register(
    name="general_assistant",
    version="v1",
    text=
    """ When a task has several steps and a later step needs the result of an earlier one:
1. Call only one tool per response.
2. Wait for its result before deciding the next call.
3. Use the exact returned value as the input, copied as a literal number. Never use variable names such as "result" or "previous_result".
4. Only write the final answer after every step is complete.""",

    activate=True,  )

system_prompt, prompt_version = default_registry.get("general_assistant")
print(f"Using prompt 'general_assistant' version={prompt_version}")


agent = Agent(system_prompt=system_prompt)

task = "Convert 100 Celsius to Fahrenheit, then calculate half of that result."
print(f"USER TASK: {task}\n")

answer = agent.run(task)

print(f"\n{'#' * 60}")
print("FINAL ANSWER:")
print(answer)
print(f"{'#' * 60}")

# This is the payoff of Module 12: one line gives you full visibility
# into cost and cache efficiency for this run.
print(f"\n[STATS] {agent.stats()}")

Using prompt 'general_assistant' version=v1
USER TASK: Convert 100 Celsius to Fahrenheit, then calculate half of that result.

[Agent] turn 1: sending 2 messages...
[Agent] turn 1: calling tool 'celsius_to_fahrenheit'
[Agent] turn 1: tool {"celsius":100} -> 'celsius_to_fahrenheit' -> 212.0
[Agent] turn 2: sending 4 messages...
[Agent] turn 2: calling tool 'calculate'
[Agent] turn 2: tool {"expression":"212.0 / 2"} -> 'calculate' -> 106.0
[Agent] turn 3: sending 6 messages...
[Agent] turn 3: final answer produced.

############################################################
FINAL ANSWER:
The result of converting 100 Celsius to Fahrenheit and then calculating half of that result is 106.0.
############################################################

[STATS] 3 LLM call(s), 1191 input + 79 output tokens, ~$0.000226 estimated cost | 0 hit(s), 3 miss(es) (0% hit rate)
